# 04 真实 COF 数据：为什么不能拿到表格就直接训练？

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wanteen/COF-ML-Tutorial/blob/main/notebooks/04_real_cof_dataset.ipynb)

这一章的目标不是训练模型，而是学会先检查数据。真实科研数据常常存在缺失值、重复记录、单位不一致和字段含义不清的问题。

In [ ]:
import pandas as pd
url='https://raw.githubusercontent.com/Wanteen/CURATED-COFs/master/cof-frameworks.csv'
df=pd.read_csv(url)
print('rows, columns =',df.shape)
display(df.head())

## 1. 先问两个最基本的问题
- 每一行代表什么？通常是一条 COF 记录。
- 每一列代表什么？可能是 ID、结构信息、性质、论文来源或其他 metadata。

如果这两个问题都回答不清楚，就不应该进入机器学习。

In [ ]:
print(df.columns.tolist())
summary=pd.DataFrame({
 'dtype':df.dtypes.astype(str),
 'missing':df.isna().sum(),
 'missing_%':(100*df.isna().mean()).round(1),
 'unique':df.nunique(dropna=True)
})
display(summary.head(30))

## 2. 什么是 missing value？
缺失值表示数据库这一格没有有效数据。它不一定等于 0。

例如：某 COF 没有记录孔径，不代表孔径为 0 Å。把缺失值直接填成 0 可能引入错误物理含义。

In [ ]:
display(summary.sort_values('missing_%',ascending=False).head(15))

## 3. Metadata 和 feature 不一样
`paper DOI`、数据库编号、文件名等信息很重要，因为它们帮助追踪数据来源；但它们通常不是材料本身的物理 feature。

**metadata（元数据）** 可以保留用于追踪，不代表应该直接作为模型输入。

## 4. 为什么要检查重复？
如果同一个 COF 或几乎相同的 COF 同时出现在 training set 和 test set，模型测试时相当于见过很接近的答案，结果可能过于乐观。

这也是后面会讲的 **data leakage（数据泄漏）** 的一种来源。

## 5. 同一个性质也可能不能直接合并
例如 CO₂ uptake 会依赖温度、压力和实验/计算方法。

因此在真实科研中，target 不只是“一个数字”，还要确认这个数字是否来自可比较的条件。

## 本章术语表
- missing value：缺失值；
- metadata：元数据，用于描述数据来源/身份；
- duplicate：重复记录；
- data leakage：测试信息不合理进入训练过程；
- provenance：数据来源与生成过程。

## Exercises
1. 找出缺失率最高的 10 列。
2. 选 5 列并判断它们更像 feature、target 还是 metadata。
3. 解释为什么“缺失”不能自动等于 0。
4. 解释为什么重复 COF 跨 train/test 会让结果变得不可靠。

### 本章最低要求
拿到真实材料数据后，先检查字段、缺失、重复、单位和条件，再考虑训练模型。